### All days of the challange:

* [Day 1: Handling missing values](https://www.kaggle.com/rtatman/data-cleaning-challenge-handling-missing-values)
* [Day 2: Scaling and normalization](https://www.kaggle.com/rtatman/data-cleaning-challenge-scale-and-normalize-data)
* [Day 3: Parsing dates](https://www.kaggle.com/rtatman/data-cleaning-challenge-parsing-dates/)
* [Day 4: Character encodings](https://www.kaggle.com/rtatman/data-cleaning-challenge-character-encodings/)
* [Day 5: Inconsistent Data Entry](https://www.kaggle.com/rtatman/data-cleaning-challenge-inconsistent-data-entry/)
___

Chào mừng bạn đến với ngày thứ 5 của Thử thách Dữ liệu 5 Ngày (5-Day Data Challenge)! (Bạn có tin được là đã trôi qua 5 ngày rồi không?) Hôm nay, chúng ta sẽ học cách làm sạch các dữ liệu văn bản không đồng nhất. Để bắt đầu, hãy nhấp vào nút "Fork Notebook" màu xanh dương ở góc trên bên phải. Thao tác này sẽ tạo một bản sao riêng tư của notebook này để bạn có thể chỉnh sửa và thực hành. Sau khi hoàn thành các bài tập, bạn có thể chọn công khai notebook của mình để chia sẻ với người khác. :)

> **Đến lượt bạn!** Trong quá trình làm việc với notebook này, bạn sẽ thấy một số ô (khối chứa mã code hoặc văn bản) có dòng chữ "Your Turn!" (Đến lượt bạn!). Đây là các bài tập giúp bạn củng cố kiến ​​thức về những khái niệm mà chúng ta đang thảo luận. Sau khi viết mã để trả lời một câu hỏi cụ thể, bạn có thể chạy mã đó bằng cách nhấp vào bên trong ô chứa mã muốn chạy rồi nhấn tổ hợp phím CTRL + ENTER (hoặc CMD + ENTER trên máy Mac). Bạn cũng có thể nhấp vào ô đó rồi nhấp vào biểu tượng mũi tên "play" (chạy) nằm ở phía bên trái đoạn mã. Nếu muốn chạy toàn bộ mã trong notebook, bạn có thể sử dụng biểu tượng mũi tên kép "fast forward" (tua nhanh) ở cuối trình chỉnh sửa notebook.

Dưới đây là những gì chúng ta sẽ thực hiện trong ngày hôm nay:

* [Get our environment set up](#Get-our-environment-set-up)
* [Do some preliminary text pre-processing](#Do-some-preliminary-text-pre-processing)
* [Use fuzzy matching to correct inconsistent data entry](#Use-fuzzy-matching-to-correct-inconsistent-data-entry)


Let's get started!

# Thiết lập môi trường làm việc
________

Việc đầu tiên cần làm là tải các thư viện mà chúng ta sẽ sử dụng. Tuy nhiên, chúng ta chưa tải các bộ dữ liệu (dataset) vào lúc này đâu nhé – việc đó sẽ được thực hiện sau!

> **Quan trọng!** Hãy đảm bảo bạn tự chạy ô mã này, nếu không phần còn lại của mã nguồn sẽ không hoạt động được!

In [2]:
# modules we'll use
import pandas as pd
import numpy as np

# helpful modules
import fuzzywuzzy
from fuzzywuzzy import process
import chardet

# set seed for reproducibility
np.random.seed(0)

Khi tôi thử đọc tệp `PakistanSuicideAttacks Ver 11 (30-November-2017).csv` lần đầu tiên, tôi gặp lỗi mã hóa ký tự, vì vậy tôi sẽ kiểm tra nhanh xem mã hóa chuẩn xác phải là gì...

In [3]:
# look at the first ten thousand bytes to guess the character encoding
with open("../input/PakistanSuicideAttacks Ver 11 (30-November-2017).csv", 'rb') as rawdata:
    result = chardet.detect(rawdata.read(100000))

# check what the character encoding might be
print(result)

{'encoding': 'Windows-1252', 'confidence': 0.73, 'language': ''}


And then read it in with the correct encoding. (If this look unfamiliar to you, check out [yesterday's challenge](https://www.kaggle.com/rtatman/data-cleaning-challenge-character-encodings/).) 

In [4]:
# read in our dat
suicide_attacks = pd.read_csv("../input/PakistanSuicideAttacks Ver 11 (30-November-2017).csv", 
                              encoding='Windows-1252')

Giờ thì chúng ta đã sẵn sàng bắt đầu! Như thường lệ, bạn có thể dành chút thời gian xem qua dữ liệu để làm quen với nó. :)


# Thực hiện bước tiền xử lý văn bản sơ bộ
___

Trong bài tập này, tôi muốn làm sạch cột "City" (Thành phố) để đảm bảo không có sự thiếu nhất quán nào trong quá trình nhập liệu. Tất nhiên, chúng ta có thể kiểm tra thủ công từng dòng và tự sửa lại những chỗ chưa thống nhất khi phát hiện ra chúng. Tuy nhiên, vẫn có một cách hiệu quả hơn để thực hiện việc này!

In [5]:
# get all the unique values in the 'City' column
cities = suicide_attacks['City'].unique()

# sort them alphabetically and then take a closer look
cities.sort()
cities

array(['ATTOCK', 'Attock ', 'Bajaur Agency', 'Bannu', 'Bhakkar ', 'Buner',
       'Chakwal ', 'Chaman', 'Charsadda', 'Charsadda ', 'D. I Khan',
       'D.G Khan', 'D.G Khan ', 'D.I Khan', 'D.I Khan ', 'Dara Adam Khel',
       'Dara Adam khel', 'Fateh Jang', 'Ghallanai, Mohmand Agency ',
       'Gujrat', 'Hangu', 'Haripur', 'Hayatabad', 'Islamabad',
       'Islamabad ', 'Jacobabad', 'KURRAM AGENCY', 'Karachi', 'Karachi ',
       'Karak', 'Khanewal', 'Khuzdar', 'Khyber Agency', 'Khyber Agency ',
       'Kohat', 'Kohat ', 'Kuram Agency ', 'Lahore', 'Lahore ',
       'Lakki Marwat', 'Lakki marwat', 'Lasbela', 'Lower Dir', 'MULTAN',
       'Malakand ', 'Mansehra', 'Mardan', 'Mohmand Agency',
       'Mohmand Agency ', 'Mohmand agency', 'Mosal Kor, Mohmand Agency',
       'Multan', 'Muzaffarabad', 'North Waziristan', 'North waziristan',
       'Nowshehra', 'Orakzai Agency', 'Peshawar', 'Peshawar ', 'Pishin',
       'Poonch', 'Quetta', 'Quetta ', 'Rawalpindi', 'Sargodha',
       'Sehwan town',

Chỉ cần nhìn qua là tôi đã thấy một số vấn đề do việc nhập liệu không đồng nhất: chẳng hạn như 'Lahore' và 'Lahore ', hay 'Lakki Marwat' và 'Lakki marwat'.

Việc đầu tiên tôi sẽ làm là chuyển tất cả về dạng chữ thường (tôi có thể đổi lại sau nếu muốn) và loại bỏ các khoảng trắng thừa ở đầu và cuối ô dữ liệu. Sự thiếu đồng nhất về cách viết hoa và các khoảng trắng thừa là những vấn đề rất phổ biến trong dữ liệu dạng văn bản; chỉ cần thực hiện các bước này, bạn có thể khắc phục tới 80% các lỗi không đồng nhất đó.

In [6]:
# convert to lower case
suicide_attacks['City'] = suicide_attacks['City'].str.lower()
# remove trailing white spaces
suicide_attacks['City'] = suicide_attacks['City'].str.strip()

Tiếp theo, chúng ta sẽ giải quyết vấn đề thiếu nhất quán phức tạp hơn.

In [7]:
# Your turn! Take a look at all the unique values in the "Province" column. 
# Then convert the column to lowercase and remove any trailing white spaces
provinces = suicide_attacks['Province'].unique()

# sort them alphabetically and then take a closer look
provinces.sort()
provinces

array(['AJK', 'Balochistan', 'Baluchistan', 'Capital', 'FATA', 'Fata',
       'KPK', 'Punjab', 'Sindh'], dtype=object)

In [8]:
# convert to lower case
suicide_attacks['Province'] = suicide_attacks['Province'].str.lower()
# remove trailing white spaces
suicide_attacks['Province'] = suicide_attacks['Province'].str.strip()

# Sử dụng phương pháp khớp mờ (fuzzy matching) để sửa lỗi nhập liệu thiếu nhất quán
___

Được rồi, hãy xem lại cột thành phố và kiểm tra xem có cần thực hiện thêm bước làm sạch dữ liệu nào nữa không.

In [9]:
# get all the unique values in the 'City' column
cities = suicide_attacks['City'].unique()

# sort them alphabetically and then take a closer look
cities.sort()
cities

array(['attock', 'bajaur agency', 'bannu', 'bhakkar', 'buner', 'chakwal',
       'chaman', 'charsadda', 'd. i khan', 'd.g khan', 'd.i khan',
       'dara adam khel', 'fateh jang', 'ghallanai, mohmand agency',
       'gujrat', 'hangu', 'haripur', 'hayatabad', 'islamabad',
       'jacobabad', 'karachi', 'karak', 'khanewal', 'khuzdar',
       'khyber agency', 'kohat', 'kuram agency', 'kurram agency',
       'lahore', 'lakki marwat', 'lasbela', 'lower dir', 'malakand',
       'mansehra', 'mardan', 'mohmand agency',
       'mosal kor, mohmand agency', 'multan', 'muzaffarabad',
       'north waziristan', 'nowshehra', 'orakzai agency', 'peshawar',
       'pishin', 'poonch', 'quetta', 'rawalpindi', 'sargodha',
       'sehwan town', 'shabqadar-charsadda', 'shangla', 'shikarpur',
       'sialkot', 'south waziristan', 'sudhanoti', 'sukkur', 'swabi',
       'swat', 'taftan', 'tangi, charsadda district', 'tank', 'taunsa',
       'tirah valley', 'totalai', 'upper dir', 'wagah', 'zhob'],
      dtype=

Có vẻ như vẫn còn một số điểm chưa đồng nhất: 'd. i khan' và 'd.i khan' có lẽ nên được coi là cùng một địa danh. (Tôi đã [tra cứu](https://en.wikipedia.org/wiki/List_of_most_populous_cities_in_Pakistan) và thấy 'd.g khan' là một thành phố riêng biệt, vì vậy tôi sẽ không gộp chúng lại với nhau.)

Tôi sẽ sử dụng thư viện [fuzzywuzzy](https://github.com/seatgeek/fuzzywuzzy) để giúp xác định các chuỗi văn bản nào có độ tương đồng cao nhất với nhau. Tập dữ liệu này đủ nhỏ để chúng ta có thể sửa lỗi thủ công, nhưng cách tiếp cận đó lại không hiệu quả khi quy mô dữ liệu lớn hơn. (Bạn có muốn sửa thủ công cả nghìn lỗi không? Còn mười nghìn lỗi thì sao? Tự động hóa quy trình càng sớm càng tốt thường là một ý hay. Hơn nữa, làm vậy cũng rất thú vị! :)

> **Khớp mờ (Fuzzy matching):** Là quá trình tự động tìm kiếm các chuỗi văn bản có độ tương đồng cao với chuỗi mục tiêu. Nhìn chung, một chuỗi được coi là "gần" với một chuỗi khác hơn nếu số lượng ký tự cần thay đổi để chuyển đổi chuỗi này thành chuỗi kia là ít hơn. Ví dụ: "apple" và "snapple" cách nhau hai bước thay đổi (thêm chữ "s" và "n"), trong khi "in" và "on" chỉ cách nhau một bước thay đổi (thay chữ "i" bằng chữ "o"). Bạn không thể luôn dựa hoàn toàn vào phương pháp khớp mờ, nhưng nó thường sẽ giúp bạn tiết kiệm được ít nhiều thời gian.

Fuzzywuzzy sẽ trả về một tỷ lệ (điểm số) khi so sánh hai chuỗi. Tỷ lệ này càng gần mức 100 thì khoảng cách chỉnh sửa giữa hai chuỗi càng nhỏ. Ở đây, chúng ta sẽ tìm ra mười chuỗi trong danh sách thành phố có khoảng cách gần nhất với chuỗi "d.i khan".

In [10]:
# get the top 10 closest matches to "d.i khan"
matches = fuzzywuzzy.process.extract("d.i khan", cities, limit=10, scorer=fuzzywuzzy.fuzz.token_sort_ratio)

# take a look at them
matches

[('d. i khan', 100),
 ('d.i khan', 100),
 ('d.g khan', 88),
 ('khanewal', 50),
 ('sudhanoti', 47),
 ('hangu', 46),
 ('kohat', 46),
 ('dara adam khel', 45),
 ('chaman', 43),
 ('mardan', 43)]

Ta có thể thấy hai mục trong danh sách thành phố rất giống với "d.i khan": đó là "d. i khan" và "d.i khan". Ngoài ra, còn có "d.g khan" – một thành phố riêng biệt – với tỷ lệ tương đồng là 88. Vì không muốn thay thế "d.g khan" bằng "d.i khan", ta sẽ thay thế tất cả các dòng trong cột Thành phố có tỷ lệ tương đồng lớn hơn 90 bằng giá trị "d. i khan".

Để thực hiện việc này, tôi sẽ viết một hàm xử lý. (Viết một hàm đa năng để tái sử dụng là ý tưởng hay nếu bạn dự đoán mình sẽ phải thực hiện một tác vụ cụ thể nhiều lần. Cách làm này giúp tránh việc sao chép và dán mã nguồn quá thường xuyên, từ đó tiết kiệm thời gian và hạn chế sai sót.)

In [11]:
# function to replace rows in the provided column of the provided dataframe
# that match the provided string above the provided ratio with the provided string
def replace_matches_in_column(df, column, string_to_match, min_ratio = 90):
    # get a list of unique strings
    strings = df[column].unique()
    
    # get the top 10 closest matches to our input string
    matches = fuzzywuzzy.process.extract(string_to_match, strings, 
                                         limit=10, scorer=fuzzywuzzy.fuzz.token_sort_ratio)

    # only get matches with a ratio > 90
    close_matches = [matches[0] for matches in matches if matches[1] >= min_ratio]

    # get the rows of all the close matches in our dataframe
    rows_with_matches = df[column].isin(close_matches)

    # replace all rows with close matches with the input matches 
    df.loc[rows_with_matches, column] = string_to_match
    
    # let us know the function's done
    print("All done!")

Giờ thì chúng ta đã có một hàm số, hãy cùng thử nghiệm nó nhé!

In [12]:
# use the function we just wrote to replace close matches to "d.i khan" with "d.i khan"
replace_matches_in_column(df=suicide_attacks, column='City', string_to_match="d.i khan")

All done!


Bây giờ, hãy kiểm tra lại các giá trị duy nhất trong cột City và đảm bảo rằng chúng ta đã xử lý mục "d.i khan" một cách chính xác.

In [13]:
# get all the unique values in the 'City' column
cities = suicide_attacks['City'].unique()

# sort them alphabetically and then take a closer look
cities.sort()
cities

array(['attock', 'bajaur agency', 'bannu', 'bhakkar', 'buner', 'chakwal',
       'chaman', 'charsadda', 'd.g khan', 'd.i khan', 'dara adam khel',
       'fateh jang', 'ghallanai, mohmand agency', 'gujrat', 'hangu',
       'haripur', 'hayatabad', 'islamabad', 'jacobabad', 'karachi',
       'karak', 'khanewal', 'khuzdar', 'khyber agency', 'kohat',
       'kuram agency', 'kurram agency', 'lahore', 'lakki marwat',
       'lasbela', 'lower dir', 'malakand', 'mansehra', 'mardan',
       'mohmand agency', 'mosal kor, mohmand agency', 'multan',
       'muzaffarabad', 'north waziristan', 'nowshehra', 'orakzai agency',
       'peshawar', 'pishin', 'poonch', 'quetta', 'rawalpindi', 'sargodha',
       'sehwan town', 'shabqadar-charsadda', 'shangla', 'shikarpur',
       'sialkot', 'south waziristan', 'sudhanoti', 'sukkur', 'swabi',
       'swat', 'taftan', 'tangi, charsadda district', 'tank', 'taunsa',
       'tirah valley', 'totalai', 'upper dir', 'wagah', 'zhob'],
      dtype=object)

Excellent! Now we only have "d.i khan" in our dataframe and we didn't have to change anything by hand. 

In [14]:
# Your turn! It looks like 'kuram agency' and 'kurram agency' should
# be the same city. Correct the dataframe so that they are.
replace_matches_in_column(df=suicide_attacks, column='City', string_to_match="kuram agency")

All done!


In [15]:
# get all the unique values in the 'City' column
cities = suicide_attacks['City'].unique()

# sort them alphabetically and then take a closer look
cities.sort()
cities

array(['attock', 'bajaur agency', 'bannu', 'bhakkar', 'buner', 'chakwal',
       'chaman', 'charsadda', 'd.g khan', 'd.i khan', 'dara adam khel',
       'fateh jang', 'ghallanai, mohmand agency', 'gujrat', 'hangu',
       'haripur', 'hayatabad', 'islamabad', 'jacobabad', 'karachi',
       'karak', 'khanewal', 'khuzdar', 'khyber agency', 'kohat',
       'kuram agency', 'lahore', 'lakki marwat', 'lasbela', 'lower dir',
       'malakand', 'mansehra', 'mardan', 'mohmand agency',
       'mosal kor, mohmand agency', 'multan', 'muzaffarabad',
       'north waziristan', 'nowshehra', 'orakzai agency', 'peshawar',
       'pishin', 'poonch', 'quetta', 'rawalpindi', 'sargodha',
       'sehwan town', 'shabqadar-charsadda', 'shangla', 'shikarpur',
       'sialkot', 'south waziristan', 'sudhanoti', 'sukkur', 'swabi',
       'swat', 'taftan', 'tangi, charsadda district', 'tank', 'taunsa',
       'tirah valley', 'totalai', 'upper dir', 'wagah', 'zhob'],
      dtype=object)

And that's it for today! If you have any questions, be sure to post them in the comments below or [on the forums](https://www.kaggle.com/questions-and-answers). 

Remember that your notebook is private by default, and in order to share it with other people or ask for help with it, you'll need to make it public. First, you'll need to save a version of your notebook that shows your current work by hitting the "Commit & Run" button. (Your work is saved automatically, but versioning your work lets you go back and look at what it was like at the point you saved it. It also lets you share a nice compiled notebook instead of just the raw code.) Then, once your notebook is finished running, you can go to the Settings tab in the panel to the left (you may have to expand it by hitting the [<] button next to the "Commit & Run" button) and setting the "Visibility" dropdown to "Public".

# More practice!
___

Do any other columns in this dataframe have inconsistent data entry? If you can find any, try to tidy them up.

You can also try reading in the `PakistanSuicideAttacks Ver 6 (10-October-2017).csv` file from this dataset and tidying up any inconsistent columns in that data file.